# Start

In [5]:
# Taruh ini di sel paling atas sendiri di Notebook-mu
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## sensor_simulation.py

In [6]:
%%writefile sensor_simulation.py
from datetime import datetime, timezone, timedelta
import random

def sensor_simulation() -> dict[str,float]:
  tz_wib = timezone(timedelta(hours=7))
  temperature = random.uniform(20.0, 40.0)
  air_humidity = random.uniform(40.0, 100.0)
  soil_moisture = random.uniform(0.0, 100.0)
  soil_ph = random.uniform(4.0, 7.0)

  return {
      'reading_timestamp': datetime.now(tz_wib).isoformat(),
      'temperature': temperature,
      'air_humidity': air_humidity,
      'soil_moisture': soil_moisture,
      'soil_ph': soil_ph

      ## if prefer lower decimal count to save space
      # round('temperature': temperature, 3),
      # round('air_humidity': air_humidity, 3),
      # round('soil_moisture': soil_moisture, 3),
      # round('soil_ph': soil_ph, 3)
  }

Overwriting sensor_simulation.py


In [7]:
from sensor_simulation import sensor_simulation

print(sensor_simulation())

{'reading_timestamp': '2026-05-31T17:13:05.835711+07:00', 'temperature': 35.53321237918743, 'air_humidity': 64.25484066640458, 'soil_moisture': 10.897093618392406, 'soil_ph': 5.241503156983485}


In [8]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print(AESGCM.generate_key(bit_length=256))

b'RHH\xf1\x97\xa5f3N\xc8\xb1\xbf\xc6,\xac\xfcFD\xffI\xa2\xa5\xbe\xc9\x19\xdfT\xf4(\x02W\xc7'


## rsa_encryption.py

In [9]:
%%writefile rsa_encryption.py
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from cryptography.hazmat.primitives import hashes

def rsa_encrypt(message: bytes, public_key_pem: str) -> bytes:
    public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

    ciphertext = public_key.encrypt(
        message,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return ciphertext

Writing rsa_encryption.py


## build_payload.py

In [47]:
%%writefile build_payload.py
from datetime import datetime, timezone, timedelta
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
import json
from rsa_encryption import rsa_encrypt
from sensor_simulation import sensor_simulation
import sqlite3
import requests

class EdgeGateway:
	def __init__(self, sensor_id, return_aad_bytes: bool = False):
		self.sensor_id = sensor_id
		self.return_aad_bytes = return_aad_bytes
		self.db_conn = sqlite3.connect(f"sensor{sensor_id}.db")
		cursor_init = self.db_conn.cursor()
		# cursor_init.execute("""
		# 	CREATE TABLE IF NOT EXISTS self_id (
		# 	id INTEGER PRIMARY KEY)
		# """)
		cursor_init.execute("""
			CREATE TABLE IF NOT EXISTS session_keys (
			id			INTEGER	PRIMARY KEY,
			key 		TEXT	NOT NULL,
			times_used	INTEGER	NOT NULL)
		""")
		cursor_init.execute("""
			CREATE TABLE IF NOT EXISTS sensor_readings (
			id			INTEGER	PRIMARY KEY,
			data		TEXT	NOT NULL,
			sent_status	INTEGER	NOT NULL)
		""")
		# cursor_init.execute("INSERT OR IGNORE INTO self_id (id) VALUES (?)", (sensor_id,))
		self.db_conn.commit()

	# def get_latest_session_key(self):
	# 	cursor = self.db_conn.cursor()
	# 	cursor.execute("""
	# 			SELECT id, key FROM session_keys ORDER BY id DESC LIMIT 1
	# 		""")
	# 	return cursor.fetchone()

	def get_latest_session_key(self):
		return self.db_conn.execute("""
				SELECT id, key FROM session_keys ORDER BY id DESC LIMIT 1
			""").fetchone()

	def build_payload(self, public_key_pem: str, send_no_session_key: bool = False) -> dict:
		tz_wib = timezone(timedelta(hours=7))

		# this one gonna cascade depending on the result. None is not a valid session_key. You need to use the current session_key
		# session_key = get_session_key() # -> raw_bytes
		# session_key for testing because get_session_key isn't finished yet
		if send_no_session_key and (row := self.get_latest_session_key()):
			session_key_id, session_key = row
		else:
			session_key_id = 1
			session_key = AESGCM.generate_key(bit_length=256)

		encrypted_session_key = rsa_encrypt(session_key, public_key_pem) # -> raw_bytes

		aad = {
			'sensor_id': 1,
			'transmission_timestamp': datetime.now(tz_wib).isoformat(),
			'encrypted_session_key': base64.b64encode(encrypted_session_key).decode('utf-8') if not send_no_session_key else None,
		}
		aad_bytes = json.dumps(aad, separators=(',', ':'), sort_keys=True).encode('utf-8')

		aesgcm = AESGCM(session_key)
		iv = os.urandom(12)
		plaintext_string = json.dumps(sensor_simulation())# -> dict[str,float]
		

		self.db_conn.execute("""
			INSERT INTO sensor_readings (data, sent_status)
			VALUES (?, 0)
		""", (plaintext_string,))
		self.db_conn.commit()

		plaintext_string = self.db_conn.execute("""
			SELECT data FROM sensor_readings
			WHERE sent_status = 0
			ORDER BY id ASC
			LIMIT 1
		""").fetchone()[0]

		plaintext_bytes = plaintext_string.encode('utf-8') 
		encrypted_raw = aesgcm.encrypt(iv, plaintext_bytes, aad_bytes)
		ciphertext = encrypted_raw[:-16]
		tag = encrypted_raw[-16:]

		# cursor = self.db_conn.cursor()
		# cursor.execute("""
		# 	UPDATE session_keys 
		# 	SET times_used = times_used + 1
		# 	WHERE id = ?
		# """, (session_key_id,))

		session_key_id_already_exist = self.db_conn.execute("""
			SELECT 1 FROM session_keys
			WHERE id = ?
		""", (session_key_id,)).fetchone()

		if session_key_id_already_exist:
			self.db_conn.execute("""
				UPDATE session_keys 
				SET times_used = times_used + 1
				WHERE id = ?
			""", (session_key_id,))
		else:
			self.db_conn.execute("""
				INSERT INTO session_keys
				(key, times_used)
				VALUES (?, 1)
			""", (session_key,))

		self.db_conn.commit()
		
		return {
			'aad': aad if not self.return_aad_bytes else aad_bytes.decode('utf-8'),
			'nonce': base64.b64encode(iv).decode('utf-8'),
			'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
			'tag': base64.b64encode(tag).decode('utf-8')
		}
	
	def post(self, url: str, *, json: dict):
		response=requests.post(url, json=json)
		if response.status_code == 200:
			self.db_conn.execute("""
				UPDATE sensor_readings
				SET sent_status = 1
				WHERE id = (
					SELECT id
					FROM sensor_readings
					WHERE sent_status = 0
					ORDER BY id ASC
					LIMIT 1
				)
			""")
			self.db_conn.commit()
		return response
		

Overwriting build_payload.py


## build_payload2.py (send raw aad)

In [11]:
%%writefile build_payload2.py
from datetime import datetime, timezone, timedelta
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
import json
from rsa_encryption import rsa_encrypt
from sensor_simulation import sensor_simulation

# def get_session_key() -> bytes | None:
#   if session_key_use_count > session_key_lifetime:
#	 return AESGCM.generate_key(bit_length=256)
#   else:
#	 return None



def build_payload2(public_key_pem: str) -> dict:
	tz_wib = timezone(timedelta(hours=7))

	# this one gonna cascade depending on the result. None is not a valid session_key. You need to use the current session_key
	# session_key = get_session_key() # -> raw_bytes
	# session_key for testing because get_session_key isn't finished yet
	session_key = b'\xf8\xb1\xab)4\xac\xdf\xfa\x8b)\xbf\xd9\xd9^c\xc5\x17\x11\x8f\xfe{\xeb\x00u\xaa9\xac:\x8aVp\xcd'
	encrypted_session_key = rsa_encrypt(session_key, public_key_pem) # -> raw_bytes

	aesgcm = AESGCM(session_key)
	iv = os.urandom(12)
	plaintext_bytes = json.dumps(sensor_simulation()).encode('utf-8') # -> dict[str,float]
	aad = {
		'sensor_id': 1,
		'transmission_timestamp': datetime.now(tz_wib).isoformat(),
		'encrypted_session_key': base64.b64encode(encrypted_session_key).decode('utf-8'),
	}
	aad_bytes = json.dumps(aad).encode('utf-8')
	# aad_bytes = json.dumps(aad, separators=(',', ':'), sort_keys=True).encode('utf-8')

	encrypted_raw = aesgcm.encrypt(iv, plaintext_bytes, aad_bytes)
	ciphertext = encrypted_raw[:-16]
	tag = encrypted_raw[-16:]

	# print('aad_bytes ', aad_bytes)
	# print('repr(aad_bytes) ', repr(aad_bytes))
	# return {
	# 	**aad,
	# 	'nonce': base64.b64encode(iv).decode('utf-8'),
	# 	'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
	# 	'tag': base64.b64encode(tag).decode('utf-8')
	# }

	return {
		'aad': aad_bytes.decode('utf-8'),
		'nonce': base64.b64encode(iv).decode('utf-8'),
		'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
		'tag': base64.b64encode(tag).decode('utf-8')
	}

Writing build_payload2.py


## get_public_key.py

In [12]:
%%writefile get_public_key.py
import requests
from build_payload import build_payload
# import json

def get_public_key() -> str:
    url='http://localhost:3000/public-key'
    response = requests.get(url)
    response.raise_for_status()
    return response.json()['public_key']
    # return response

# server_public_key = get_server_public_key()
# print(server_public_key)

Writing get_public_key.py


In [13]:
# from cryptography.hazmat.primitives.serialization import load_pem_public_key

# # 1. String public key mentah (misal hasil ambil dari Node.js kemarin)
# public_key_pem = server_public_key

# # 2. Import fungsinya dan ubah string menjadi objek objek kunci beneran
# public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

# # Sekarang variabel 'public_key' sudah menjadi objek RSA asli 
# # yang siap digunakan untuk enkripsi data!
# print(public_key)
# # Output: <class 'cryptography.hazmat.backends.openssl.rsa._RSAPublicKey'>

## Payload content

In [14]:
# from payload import build_payload

# print(build_payload(server_public_key))

## Send payload

In [49]:
import requests
import time
from build_payload import EdgeGateway
# from build_payload2 import build_payload2
from get_public_key import get_public_key
import numpy as np
from scipy import stats

server_public_key = get_public_key()

url = 'http://localhost:3000/telemetry'

sensor1 = EdgeGateway(1)

# CPU warm up
# for _ in range(10):
#     sensor1.build_payload(server_public_key)
    
durations1 = []

for _ in range(1):
    start = time.perf_counter()
    payload = sensor1.build_payload(server_public_key)
    print(payload)
    
    response=sensor1.post(url, json=payload)
    end = time.perf_counter()
    durations1.append(end - start)

print(response, response.json())
# ===================================================================================================================================================================================================================================================================================================================================================================================================================
# url2 = 'http://localhost:3000/telemetry2'
# durations2 = []

# for _ in range(10):
#     build_payload2(server_public_key)

# for _ in range(1):
#     start = time.perf_counter()
#     payload = build_payload2(server_public_key)
    
#     response=requests.post(url2, json=payload)
#     end = time.perf_counter()
#     durations2.append(end - start)


# # Statistik dasar
d1 = np.array(durations1) * 1000
# d2 = np.array(durations2) * 1000

print(f"Metode 1 - median: {np.median(d1):.3f}ms, std: {np.std(d1):.3f}ms")
# print(f"Metode 2 - median: {np.median(d2):.3f}ms, std: {np.std(d2):.3f}ms")

# # T-test untuk cek signifikansi
# t_stat, p_value = stats.ttest_ind(d1, d2)
# print(f"p-value: {p_value:.4f}")

# if p_value < 0.05:
#     print("Perbedaan SIGNIFIKAN")
# else:
#     print("Perbedaan TIDAK signifikan")
    
# durations.sort()
# print(f"median: {durations[len(durations)//2]*1000:.3f}ms")
# print(f"min: {durations[0]*1000:.3f}ms")
# print(f"max: {durations[-1]*1000:.3f}ms")
# print(f"mean: {sum(durations)/len(durations)*1000:.3f}ms")

AttributeError: 'bytes' object has no attribute 'encode'

# Stop

In [1]:
import json

aad = {
      'sensor_id': 1,
      'transmission_timestamp': 'datetime.now(tz_wib).isoformat()',
      'encrypted_session_key': 'session_key',
  }

aad_json = json.dumps(aad)
aad_bytes = aad_json.encode('utf-8')
print(aad_json)
print(aad_bytes)
print(type(aad_json))
print(type(aad_bytes))
# print(aad.encode('utf-8'))

{"sensor_id": 1, "transmission_timestamp": "datetime.now(tz_wib).isoformat()", "encrypted_session_key": "session_key"}
b'{"sensor_id": 1, "transmission_timestamp": "datetime.now(tz_wib).isoformat()", "encrypted_session_key": "session_key"}'
<class 'str'>
<class 'bytes'>


In [ ]:
import base64
import json
random_bytes = b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'
print({'random_bytes': random_bytes})
print(type(random_bytes))
A = base64.b64encode(random_bytes)
B = A.decode('utf-8')
print({'random_bytes': A})
print(type(A))
print({'random_bytes': B})
print(type(B))
myjson= json.dumps({'random_bytes' : 1})
print(myjson)
print(type(myjson))

{'random_bytes': b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'}
<class 'bytes'>
{'random_bytes': b'RIW0n6uJRZZDAuWcYksQBbojMUv39ZbYHDRRthUOC0I='}
<class 'bytes'>
{'random_bytes': 'RIW0n6uJRZZDAuWcYksQBbojMUv39ZbYHDRRthUOC0I='}
<class 'str'>
{"random_bytes": 1}
<class 'str'>


In [ ]:
mydict = {'random_bytes': B}
print(type(mydict))

<class 'dict'>


In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

session_key = AESGCM.generate_key(bit_length=256)
aesgcm = AESGCM(session_key)

aad = {
    'mode': ...,
    'algorithm': ...,
    'sensor_id': ...,
    'timestamp': ...,
    'sequence_number': ...,
    'encrypted_session_key': ...,
    # 'nonce': ...,
    # 'ciphertext': ...,
    # 'tag': ...
}

ciphertext_with_tag = aesgcm.encrypt(nonce, plaintext, aad)
ciphertext = ciphertext_with_tag[:-16]
tag = ciphertext_with_tag[-16:]

In [ ]:
import os
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# ==========================================
# 1. INSILIALISASI KUNCI ( Dilakukan sekali )
# ==========================================
# GCM membutuhkan kunci simetris minimal 128-bit (16 bytes)
# Kunci ini harus sama dan rahasia antara Sensor dan Server
session_key = AESGCM.generate_key(bit_length=128)
aesgcm = AESGCM(session_key)


# ==========================================
# 2. SISI SENSOR (PROSES ENKRIPSI & KIRIM)
# ==========================================
print("--- SISI SENSOR ---")

# a. Dapatkan Data utama dan Timestamp didapat
timestamp_didapat = datetime.now().isoformat()
nilai_sensor = "24.5 C"

# Gabungkan data utama ke dalam Plaintext (untuk dienkripsi)
plaintext = f"{nilai_sensor}|{timestamp_didapat}".encode('utf-8')

# b. Buat Timestamp Kirim dan Metadata untuk AAD (tidak dienkripsi)
timestamp_kirim = datetime.now().isoformat()
sensor_id = "SENSOR-NODE-01"

# Gabungkan metadata menjadi AAD (Additional Authenticated Data)
aad = f"{sensor_id}|{timestamp_kirim}".encode('utf-8')

# c. Buat IV (Initialization Vector)
# Sesuai rekomendasi NIST, gunakan ukuran tepat 96-bit (12 bytes)
iv = os.urandom(12)

# d. Proses Enkripsi menggunakan AES-GCM
# Fungsi ini otomatis menghasilkan Ciphertext + Authentication Tag jadi satu kesatuan
ciphertext_with_tag = aesgcm.encrypt(iv, plaintext, aad)

print(f"IV (Hex)         : {iv.hex()}")
print(f"AAD Terbuka      : {aad.decode('utf-8')}")
print(f"Ciphertext (Hex) : {ciphertext_with_tag.hex()}\n")

# [ Paket data ini kemudian dikirim melalui jaringan (misal via MQTT/HTTP) ]
# Komponen yang dikirim: iv, aad, dan ciphertext_with_tag



# ==========================================
# 3. SISI SERVER (PROSES DEKRIPSI & VERIFIKASI)
# ==========================================
print("--- SISI SERVER (MENERIMA PAKET) ---")

# Server menerima komponen paket data
paket_iv = iv
paket_aad = aad
paket_crypto = ciphertext_with_tag

try:
    # Server melakukan dekripsi sekaligus verifikasi AAD secara otomatis
    decrypted_data = aesgcm.decrypt(paket_iv, paket_crypto, paket_aad)

    # Jika lolos verifikasi, server membaca data
    # Membongkar AAD (Metadata)
    sensor_id_rec, time_kirim_rec = paket_aad.decode('utf-8').split('|')
    # Membongkar Plaintext (Data Inti)
    nilai_sensor_rec, time_didapat_rec = decrypted_data.decode('utf-8').split('|')

    print("✅ VERIFIKASI BERHASIL: Paket Otentik dan Tidak Dimanipulasi!")
    print(f"Metadata Server -> ID: {sensor_id_rec} | Dikirim: {time_kirim_rec}")
    print(f"Data Sensor     -> Nilai: {nilai_sensor_rec} | Dibaca: {time_didapat_rec}")

except Exception as e:
    # Jika peretas mengubah satu bit saja pada Ciphertext atau AAD (waktu kirim),
    # GCM akan mendeteksinya dan melemparkan error (FAIL).
    print("❌ VERIFIKASI GAGAL: Paket telah dimanipulasi atau kunci salah!")

--- SISI SENSOR ---
IV (Hex)         : 56fc69a043eb2854f549b17e
AAD Terbuka      : SENSOR-NODE-01|2026-05-21T15:50:02.923002
Ciphertext (Hex) : 60bf5c02dc34703adc582f5764e983fc16675b50fc98292ce3ebf8987586bf81c5b8a2bd832056cc561a4b8aaf84680ec6

--- SISI SERVER (MENERIMA PAKET) ---
✅ VERIFIKASI BERHASIL: Paket Otentik dan Tidak Dimanipulasi!
Metadata Server -> ID: SENSOR-NODE-01 | Dikirim: 2026-05-21T15:50:02.923002
Data Sensor     -> Nilai: 24.5 C | Dibaca: 2026-05-21T15:50:02.922861


In [ ]:
print(ciphertext_with_tag)

b'`\xbf\\\x02\xdc4p:\xdcX/Wd\xe9\x83\xfc\x16g[P\xfc\x98),\xe3\xeb\xf8\x98u\x86\xbf\x81\xc5\xb8\xa2\xbd\x83 V\xccV\x1aK\x8a\xaf\x84h\x0e\xc6'


In [ ]:
import os

def generate_aes_key():
    return os.urandom(32)  # 256-bit

print(generate_aes_key())

b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'


In [ ]:
import os
import base64

def generate_aes_key():
    return os.urandom(16)  # 256-bit

iv = generate_aes_key()
iv_64 = base64.b64encode(iv)
json_iv = iv_64.decode("utf-8")
print(iv)
print(iv_64)
print(json_iv)

b"\n\x08\xbf\x00\x93\x83d\xd5\x174\x1a\xa1'\x1e\xc8R"
b'Cgi/AJODZNUXNBqhJx7IUg=='
Cgi/AJODZNUXNBqhJx7IUg==


In [ ]:
# from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
# from cryptography.hazmat.primitives import padding
import json

data = {
        'timestamp': '2026-05-17T01:08:00Z',
        'sensor_id': '01',
        'readings': {
            'temperature': 26,
            'humidity': 93
        }
}

plaintext = json.dumps(data)
print(plaintext)

{"timestamp": "2026-05-17T01:08:00Z", "sensor_id": "01", "readings": {"temperature": 26, "humidity": 93}}


In [ ]:
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.serialization import load_pem_public_key

def rsa_encrypt(aes_key: bytes, public_key_pem) -> bytes:
    # public_key_pem = get_server_public_key()
    public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

    ciphertext = public_key.encrypt(
        aes_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return ciphertext

print(rsa_encrypt(b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'
,'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAt8naqge5WFHlloyPDMwN\n3JZoV26EPzfhdC7MkYD6MLwvOesXn2lM9PBNt8kjEyb8lBnAyWYmWA/HJ0GiP07l\n++NoIYSCJIJttQhzD+LDzUjWhf5poOWDwE7GZlAQ2Aqj+ffBdOI7D2RBN+4YlT2t\ne0UmNJ7Y8AvukGHDMW8TCN8Arp6Rx/wqI1y61AH6IfQ+igvdMjTmKGlXuMNnu+bO\nLg8ig7jy4FtLNHRxnL/LTMnDH7+rXV72cT1Rw8yAWwhOQS6D8IYsWhpJ8YnC7ghw\nHdNuUjm6SfpZKHuJyz/yg6siM3TS7xlAXVT2yktBw5YnN0Da7duOMqkODZupMjGw\nrwIDAQAB\n-----END PUBLIC KEY-----\n'))

b'\x06.\xc5B"\xcf=t\x19\xaf\x8bb\xcb5\xbc\x8c\x80\xf6\x06\xa2\xe1}$p\x96\x079\x14\tA\xa2\x1b\xf0\x1a\xb4\xe7\',\xedG>2\xe2E"m\xe8\x85\x94\xb7\xce\x18\xa3\xf3s\x93\xe7\t\xff,\x19;\x9c\x15Ih\x04.}]\x97w\x87G\x19>\xd7\xfe\xdf\x7f\xe4&v\x90\xd7\x87\r\x0b\x8a\xe0\x99&\xd2pN\xee\xba\x9f\\,y\x90\n\x8f\xa3\x01\x07\xcbU\x90_b\xfe^\xeca\xe0\xa3%\x11\xdd\xecX\xa0u\xc3\xce\x93)(\xde\x1c\x1f\x87\xe4\x1e\xa7\x97\xb2\x83\xbaT\xd7\'L\x08Y t7\xb6\x8e\x02`\x0f\x9b\n\x87\x7f\xedf\x1a\xf9#\r\x8e+\xb1n\xdf\xad\xa1\t\xe6\xfa\xb7\xddz`\x0f\x01O\x16\x11*\xc2p94g\t\xe9\x80\xc0\xe2\x05\xe9\xe1c\xdc\x84+\xb1\x89}\xeey\xa4\xe5\x0f\x9dg\x88\xcb\x8dr%\n\x96\x7f\xd1\xcc\x04\xf6\x1a^\xed:\xc5\xfd\xac/\xdb\xc2\xbaP\xd7nC^\x14\x98\xd3\xea\xf5\xbe4y9\xbe2TI\xba\x87='


In [ ]:
sensor_data = {}

def generate_sensor_data():
    # global sensor_data
    sensor_data = {42}
    print(sensor_data)

generate_sensor_data()
print(sensor_data)

{42}
{}


In [ ]:
def generate_sensor_data():
  state.sensor_data = {42}

class State:
  sensor_data = None

state = State()



generate_sensor_data()
print(state.sensor_data)

{42}
